# Chapter 3 Demo: Machine Learning and the Train-Test Split

Run each cell in order to see how a model is trained, tested, over-fitted and leaked on real data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-03-machine-learning.ipynb)

Lecture: [en-03-machine-learning](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-03-machine-learning.md)  
Labs: [LM301](https://rathachai.github.io/DA-LAB/learnings/linear/lm301.html) | [LM302](https://rathachai.github.io/DA-LAB/learnings/linear/lm302.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## (a) Load the points
The lab data `lm301_points.csv` has 40 points `(x, y)`. The `split` column already says which rows are **train** and which are **test**. Always look at the data first.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm301_points.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

Quick summary of every column.

In [ ]:
df.describe()

Scatter plot: **circles = train**, **diamonds = test**.

In [ ]:
tr = df[df.split == "train"]; te = df[df.split == "test"]
plt.figure(figsize=(6,4))
plt.scatter(tr.x, tr.y, marker="o", label="train")
plt.scatter(te.x, te.y, marker="D", label="test")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("lm301 points")
print(df.split.value_counts())

## (b) Fit a line on TRAIN only
The model learns slope and intercept from the train rows. The test rows stay hidden until grading.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(tr[["x"]], tr.y)
print("slope =", model.coef_[0].round(3), " intercept =", round(model.intercept_, 3))
xs = pd.DataFrame({"x": np.linspace(df.x.min(), df.x.max(), 50)})
plt.figure(figsize=(6,4))
plt.scatter(tr.x, tr.y, label="train"); plt.scatter(te.x, te.y, marker="D", label="test")
plt.plot(xs.x, model.predict(xs), "k-", label="fitted line"); plt.legend()

Score the same line on train and on test. A big gap would be a warning sign (see lecture section 3.2).

In [ ]:
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error,
                             mean_absolute_percentage_error, r2_score)
def scores(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": root_mean_squared_error(y, p),
            "MAPE %": 100 * mean_absolute_percentage_error(y, p), "R2": r2_score(y, p)}
res = pd.DataFrame({"train": scores(tr.y, model.predict(tr[["x"]])),
                    "test": scores(te.y, model.predict(te[["x"]]))}).T
res.round(3)

## (c) Memorising vs understanding
Take only 10 train points. Fit a **degree-1** line (understands the trend) and a **degree-9** polynomial (can memorise every point). Then check both on the test rows.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
small = tr.sample(10, random_state=1).sort_values("x")
def poly(d): return make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression())
m1 = poly(1).fit(small[["x"]], small.y); m9 = poly(9).fit(small[["x"]], small.y)
print(len(small), "training points,", len(te), "test points")
small

Plot both curves. The degree-9 curve passes through every training point but wiggles between them.

In [ ]:
g = pd.DataFrame({"x": np.linspace(df.x.min(), df.x.max(), 300)})
plt.figure(figsize=(7,4.5))
plt.scatter(small.x, small.y, label="train (10)"); plt.scatter(te.x, te.y, marker="D", label="test")
plt.plot(g.x, m1.predict(g), label="degree 1"); plt.plot(g.x, m9.predict(g), label="degree 9")
lo, hi = df.y.min(), df.y.max(); plt.ylim(lo - 0.5*(hi-lo), hi + 0.5*(hi-lo)); plt.legend()

Train vs test RMSE. The overfitting signature is a **gap**: tiny train error, large test error.

In [ ]:
rm = lambda m, d: root_mean_squared_error(d.y, m.predict(d[["x"]]))
gap = pd.DataFrame({"train RMSE": [rm(m1, small), rm(m9, small)],
                    "test RMSE": [rm(m1, te), rm(m9, te)]}, index=["degree 1", "degree 9"])
gap["gap (test - train)"] = gap["test RMSE"] - gap["train RMSE"]
gap.round(3)

## (d) How big should the train share be?
For each train ratio from 10 % to 90 %, repeat 30 random splits of all 40 points. Each time, fit on train and record train and test RMSE.

In [ ]:
from sklearn.model_selection import train_test_split, ShuffleSplit
X, y = df[["x"]], df.y
rows = []
for r in range(10, 100, 10):
    for i, j in ShuffleSplit(n_splits=30, train_size=r/100, random_state=0).split(X):
        m = LinearRegression().fit(X.iloc[i], y.iloc[i])
        rows.append({"ratio": r, "train_rmse": root_mean_squared_error(y.iloc[i], m.predict(X.iloc[i])),
                     "test_rmse": root_mean_squared_error(y.iloc[j], m.predict(X.iloc[j]))})
runs = pd.DataFrame(rows)
runs.head()

Summary per ratio: mean and standard deviation (std) of the errors over 30 splits.

In [ ]:
summ = runs.groupby("ratio").agg(train_mean=("train_rmse","mean"), test_mean=("test_rmse","mean"),
                                 test_std=("test_rmse","std"))
summ.round(3)

Mean test RMSE with a band of plus/minus one std. A **small train** set gives an unstable line; a **small test** set gives a noisy score. Both widen the band.

In [ ]:
plt.figure(figsize=(6,4))
plt.errorbar(summ.index, summ.test_mean, yerr=summ.test_std, fmt="o-", capsize=4)
plt.xlabel("train ratio (%)"); plt.ylabel("test RMSE"); plt.title("mean +/- std over 30 splits")

## (e) Seeing the instability
Plot the fitted lines from 20 random splits at 10 % train and at 80 % train.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4), sharey=True)
xs = pd.DataFrame({"x": [df.x.min(), df.x.max()]})
for a, r in zip(ax, [10, 80]):
    for seed in range(20):
        Xa, _, ya, _ = train_test_split(X, y, train_size=r/100, random_state=seed)
        a.plot(xs.x, LinearRegression().fit(Xa, ya).predict(xs), alpha=0.5)
    a.scatter(df.x, df.y, c="k", s=10); a.set_title(f"{r}% train: 20 fitted lines")

## (f) Many features: `lm302_data.csv`
Now 7 sensors `x1..x7` predict `y`. Some sensors are good, some are pure noise. Load and inspect.

In [ ]:
url2 = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm302_data.csv"
d2 = pd.read_csv(url2)
print(d2.shape, d2.split.value_counts().to_dict())
d2.head()

Summary statistics.

In [ ]:
d2.describe().round(2)

Add `ln_x5 = ln(x5)` and split by the `split` column. Show the new column.

In [ ]:
d2["ln_x5"] = np.log(d2.x5)
t2 = d2[d2.split == "train"]; s2 = d2[d2.split == "test"]
print(len(t2), "train rows,", len(s2), "test rows")
d2[["x5", "ln_x5", "split"]].head()

Compare feature sets. **Good** = x1, x2, x6, x7, ln x5. **With noise** = good + x3, x4. Look at train and test MAE and MAPE.

In [ ]:
good = ["x1", "x2", "x6", "x7", "ln_x5"]
sets = {"good only": good, "good + noise (x3,x4)": good + ["x3", "x4"]}
mape = lambda y, p: 100 * mean_absolute_percentage_error(y, p)
out = []
for name, f in sets.items():
    m = LinearRegression().fit(t2[f], t2.y)
    out.append({"features": name,
        "train MAE": mean_absolute_error(t2.y, m.predict(t2[f])), "test MAE": mean_absolute_error(s2.y, m.predict(s2[f])),
        "train MAPE %": mape(t2.y, m.predict(t2[f])), "test MAPE %": mape(s2.y, m.predict(s2[f]))})
cmp_ = pd.DataFrame(out).set_index("features")
cmp_.round(3)

The same numbers as a bar chart.

In [ ]:
cmp_[["train MAE", "test MAE"]].plot.bar(figsize=(6,4), rot=0)
plt.ylabel("MAE"); plt.title("Does noise help?")

Now use only **20 %** of the 100 rows for training (20 rows!) and repeat 30 random splits. Small training sets let noise features do more damage.

In [ ]:
rows = []
for i, j in ShuffleSplit(n_splits=30, train_size=0.2, random_state=0).split(d2):
    a, b = d2.iloc[i], d2.iloc[j]
    for name, f in sets.items():
        m = LinearRegression().fit(a[f], a.y)
        rows.append({"features": name, "train MAE": mean_absolute_error(a.y, m.predict(a[f])),
                     "test MAE": mean_absolute_error(b.y, m.predict(b[f]))})
rep = pd.DataFrame(rows).groupby("features").agg(["mean", "std"])
rep.round(3)

Plot the average test MAE with its spread.

In [ ]:
mm = rep["test MAE"]
plt.figure(figsize=(5,4)); plt.bar(mm.index, mm["mean"], yerr=mm["std"], capsize=6)
plt.ylabel("test MAE (20% train, 30 splits)"); plt.xticks(rotation=10)

## (g) Data leakage
Leakage means the test rows secretly helped build the model. Here is a classic case: **feature selection**. We simulate 100 samples with 200 features and a target `y` that is pure noise, so no model can truly predict it. Honest R2 should be about 0 or below.

In [ ]:
rng = np.random.default_rng(0)
sim = pd.DataFrame(rng.normal(size=(100, 200)), columns=[f"f{i}" for i in range(200)])
sim["y"] = rng.normal(size=100)
print(sim.shape)
sim.iloc[:, [0, 1, 2, 3, -1]].head()

Two workflows, 30 random seeds each. **A (leak):** pick the 5 features most related to `y` using ALL rows, then split. **B (honest):** split first, pick the 5 features on the train rows only. Both are scored on the same test rows.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
Xs, ys = sim.drop(columns="y"), sim.y
rows = []
for seed in range(30):
    sel_all = SelectKBest(f_regression, k=5).fit(Xs, ys)                    # A: sees test rows
    cols_a = Xs.columns[sel_all.get_support()]
    Xa, Xb, ya, yb = train_test_split(Xs, ys, train_size=0.5, random_state=seed)
    cols_b = Xs.columns[SelectKBest(f_regression, k=5).fit(Xa, ya).get_support()]   # B: train only
    r2 = lambda c: r2_score(yb, LinearRegression().fit(Xa[c], ya).predict(Xb[c]))
    rows.append({"A: select on ALL (leak)": r2(cols_a), "B: select on TRAIN only": r2(cols_b)})
lk = pd.DataFrame(rows)
lk.describe().round(3)

Mean test R2 and a box plot. A looks better (higher R2) only because the selection peeked at the test rows. B is the honest answer: no real skill. The same idea is why scalers, imputers and any other pre-processing must be fitted on the training set only.

In [ ]:
print(lk.mean().round(3))
lk.plot.box(figsize=(6,4)); plt.axhline(0, color="gray", ls="--"); plt.ylabel("test R2")

An even worse leak is giving the model the answer: add `y` itself as a feature on the lm302 data.

In [ ]:
a, b = train_test_split(d2, train_size=0.2, random_state=0)
f = good + ["x3", "x4"]
m = LinearRegression().fit(a[f + ["y"]], a.y)
print("R2 on test with y as a feature:", round(r2_score(b.y, m.predict(b[f + ["y"]])), 4))
print("R2 on test without it        :", round(r2_score(b.y, LinearRegression().fit(a[f], a.y).predict(b[f])), 4))

## (h) Key takeaways

- Training error is optimistic. Judge a model on **unseen** data.
- Overfitting shows as a **gap**: very low train error, high test error (degree 9 in part c).
- The split ratio is a trade-off: small train gives an unstable line, small test gives a noisy score.
- One split is noisy. **Repeat** the split and report mean and spread.
- Noise features hurt more when the training set is small.
- **Leakage** (feature selection or pre-processing on all data, or `y` as a feature) gives scores that look great and mean nothing. A test R2 of 1.0 is a red flag.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io